# Axis-1 RF Control + XGBoost baseline (Kaggle-mirror loader)

**Why not the repo loader?** The GitHub repo ships **no** CIC-IDS2017 data, so the
repo's `CICIDS2017().load()` falls through to its auto-download from the academic
CIC server, which on Colab returns a non-zip page → `BadZipFile`. This notebook
sidesteps that entirely: it loads CIC-IDS2017 from the **same Kaggle mirror your
Axis-2 LLM notebook uses** (`dhoogla/cicids2017`), applies the V1–V4 variant
switches (PCA × class-mapping), and runs the two classical experiments. No `src`,
no `tabpfn`, no academic-server download.

| Experiment | Reviewer | What |
|------------|----------|------|
| **A1** RF control | 4.1 | RF on the same balanced 2,000/class subsample the FMs use, vs full train |
| **A3** GBM baseline | 1.3 | XGBoost (falls back to sklearn HistGB) under the identical protocol |

**Faithfulness check (built in):** `rf_pruned_fulltrain` on $V_1$ should land at
≈0.9947 (the manuscript's Table-8 RF). If it does, the Kaggle mirror is consistent
with the data the foundation-model runs used and the R4.1 decomposition is valid.


In [ ]:
!git clone https://anonymous.4open.science/r/ai_for_ids

Cloning into 'ai_for_ids'...
remote: Enumerating objects: 508, done.
remote: Counting objects: 100% (508/508), done.
remote: Compressing objects: 100% (399/399), done.
remote: Total 508 (delta 110), reused 465 (delta 108), pack-reused 0 (from 0)
Receiving objects: 100% (508/508), 9.38 MiB | 29.83 MiB/s, done.
Resolving deltas: 100% (110/110), done.


In [ ]:
# ============================================================
# 0) CONFIG + INSTALL
# ============================================================
!pip install -q kagglehub xgboost
import os
OUT     = "rf_control_and_xgb"
FM_RUNS = "multiseed_runs"            # student's FM results (in the cloned repo) for the R4.1 decomposition
SEEDS              = [0, 1, 2, 3, 4]
BALANCED_PER_CLASS = 2000
TEST_SIZE          = 0.30
# CIC-IDS2017 variants: (pca, classes_mapping)
VARIANTS = {
    "V1": dict(pca=False, classes_mapping=False),
    "V2": dict(pca=True,  classes_mapping=False),
    "V3": dict(pca=False, classes_mapping=True),
    "V4": dict(pca=True,  classes_mapping=True),
}
os.makedirs(OUT, exist_ok=True)

# --- Kaggle credentials (Colab Secrets first, then env) ---
try:
    from google.colab import userdata
    os.environ.setdefault("KAGGLE_USERNAME", userdata.get("KAGGLE_USERNAME"))
    os.environ.setdefault("KAGGLE_KEY",      userdata.get("KAGGLE_KEY"))
except Exception:
    pass
assert os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"), \
    "Set KAGGLE_USERNAME and KAGGLE_KEY (Colab Secrets or env) — same creds as the LLM notebook."
print("Kaggle creds loaded. Output:", os.path.abspath(OUT))


Kaggle creds loaded. Output: /content/rf_control_and_xgb


In [ ]:
# ============================================================
# 1) IMPORTS
# ============================================================
import json, time, glob, random
import numpy as np, pandas as pd
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import IncrementalPCA
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
try:
    from xgboost import XGBClassifier; HAS_XGB = True
except Exception:
    from sklearn.ensemble import HistGradientBoostingClassifier; HAS_XGB = False
GBM_NAME = "xgboost" if HAS_XGB else "hist_gbm"
print("GBM backend:", GBM_NAME)

def set_seed(s): random.seed(s); np.random.seed(s)
def bootstrap_ci(x, n_boot=5000, ci=0.95, seed=0):
    x=np.asarray(x,float)
    if len(x)==0: return (np.nan,)*3
    rng=np.random.default_rng(seed)
    m=rng.choice(x,size=(n_boot,len(x)),replace=True).mean(axis=1)
    lo,hi=np.percentile(m,[100*(1-ci)/2,100*(1+ci)/2]); return float(x.mean()),float(lo),float(hi)
def balanced_subsample(df, n_per_class, seed):
    rng=np.random.default_rng(seed); out=[]
    for _,sub in df.groupby("Label"):
        out.append(sub.iloc[rng.choice(len(sub),size=min(len(sub),n_per_class),replace=False)])
    return pd.concat(out,ignore_index=True)


GBM backend: xgboost


In [ ]:
# ============================================================
# 2) CIC-IDS2017 LOADER (Kaggle mirror) + VARIANT BUILDER
# ============================================================
# Same mapping the Axis-2 LLM notebook uses, so fine/grouped taxonomies match.
CIC_MAP = {
    'BENIGN':'BENIGN','Benign':'BENIGN','DDoS':'DDoS',
    'DoS Hulk':'DoS','DoS GoldenEye':'DoS','DoS slowloris':'DoS','DoS Slowhttptest':'DoS',
    'PortScan':'Port Scan','Portscan':'Port Scan',
    'FTP-Patator':'Brute Force','SSH-Patator':'Brute Force','Bruteforce':'Brute Force',
    'Bot':'Bot',
    'Web Attack \x96 Brute Force':'Web Attack','Web Attack \x96 Sql Injection':'Web Attack','Web Attack \x96 XSS':'Web Attack',
    'Web Attack  Brute Force':'Web Attack','Web Attack  Sql Injection':'Web Attack','Web Attack  XSS':'Web Attack',
    'Web Attack':'Web Attack','Infiltration':'Infiltration','Heartbleed':'Heartbleed',
}
def _basic_clean(df):
    df=df.copy(); df.drop_duplicates(inplace=True)
    df.replace([np.inf,-np.inf],np.nan,inplace=True)
    for c in ("Flow_Bytes/s","Flow_Packets/s"):
        if c in df.columns: df[c]=df[c].fillna(df[c].median())
    df.dropna(inplace=True)
    for c in df.columns:
        if df[c].dtype==object: continue
        lo,hi=df[c].min(),df[c].max()
        if "float" in str(df[c].dtype) and lo>np.finfo(np.float32).min and hi<np.finfo(np.float32).max:
            df[c]=df[c].astype(np.float32)
    nun=df.nunique(); return df[nun[nun>1].index]

_RAW=None
def _load_raw():
    global _RAW
    if _RAW is None:
        import kagglehub
        folder=kagglehub.dataset_download("dhoogla/cicids2017")
        pqs=sorted(glob.glob(os.path.join(folder,"**","*.parquet"),recursive=True))
        df=pd.concat([pd.read_parquet(p) for p in pqs],ignore_index=True) if pqs else \
           pd.concat([pd.read_csv(c,encoding="latin-1",low_memory=False)
                      for c in sorted(glob.glob(os.path.join(folder,"**","*.csv"),recursive=True))],ignore_index=True)
        df.columns=[c.strip().replace(" ","_") for c in df.columns]
        _RAW=df
        print(f"  raw CIC mirror loaded: {df.shape}  cols incl ClassLabel={'ClassLabel' in df.columns}")
    return _RAW

_CACHE={}
def get_variant(V):
    if V in _CACHE: return _CACHE[V]
    pca, mapping = VARIANTS[V]["pca"], VARIANTS[V]["classes_mapping"]
    df=_load_raw().copy()
    if mapping and "ClassLabel" in df.columns:
        df=df.drop(columns=["Label"],errors="ignore").rename(columns={"ClassLabel":"Label"}); pre=True
    else:
        df=df.drop(columns=["ClassLabel"],errors="ignore"); pre=False
    df["Label"]=df["Label"].astype(str).str.replace(r"[^\x00-\x7F]+","",regex=True).str.strip()
    df=_basic_clean(df)
    if mapping and not pre: df["Label"]=df["Label"].map(lambda x:CIC_MAP.get(x,x))
    df["Label"]=df["Label"].replace({"Benign":"BENIGN"})
    y=df["Label"].values; X=df.drop(columns=["Label"]).select_dtypes(include=[np.number])
    if pca:
        Xs=StandardScaler().fit_transform(X.values); k=max(1,X.shape[1]//2)
        ip=IncrementalPCA(n_components=k,batch_size=500)
        for i in range(0,len(Xs),500): ip.partial_fit(Xs[i:i+500])
        Xv=ip.transform(Xs); cols=[f"PC{i+1}" for i in range(k)]
        dv=pd.DataFrame(Xv,columns=cols); dv["Label"]=y
    else:
        dv=X.copy(); dv["Label"]=y
    _CACHE[V]=dv
    print(f"  [{V}] {dv.shape}  classes={dv['Label'].nunique()}  (pca={pca}, mapping={mapping})")
    return dv


In [ ]:
# ============================================================
# 3) PER-SEED EVALUATOR  (now also captures PER-CLASS F1)
#    same split + balanced subsample as the FM protocol
# ============================================================
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support

# rare-class watch-list (covers both fine V1/V2 and grouped V3/V4 spellings)
RARE = ["Bot", "Infiltration", "Web Attack",
        "Web Attack Brute Force", "Web Attack SQL Injection",
        "Web Attack XSS", "Heartbleed"]

def multiseed_eval(dfv, builder, bound=None, label_encode=False):
    rows, perclass = [], []
    for seed in SEEDS:
        set_seed(seed)
        tr, te = train_test_split(dfv, test_size=TEST_SIZE, random_state=seed, stratify=dfv["Label"])
        if bound is not None: tr = balanced_subsample(tr, bound, seed)
        Xtr, ytr = tr.drop(columns=["Label"]).values, tr["Label"].values
        Xte, yte = te.drop(columns=["Label"]).values, te["Label"].values
        clf = builder(seed)
        if label_encode:
            le = LabelEncoder().fit(np.concatenate([ytr, yte]))
            t = time.time(); clf.fit(Xtr, le.transform(ytr)); fit = time.time() - t
            t = time.time(); yp = le.inverse_transform(clf.predict(Xte)); pr = time.time() - t
        else:
            t = time.time(); clf.fit(Xtr, ytr); fit = time.time() - t
            t = time.time(); yp = clf.predict(Xte); pr = time.time() - t
        rows.append(dict(seed=seed, accuracy=accuracy_score(yte, yp),
                         macro_f1=f1_score(yte, yp, average="macro"),
                         n_train=len(ytr), n_test=len(yte), fit_seconds=fit, predict_seconds=pr))
        # per-class precision/recall/F1 for every class this seed
        labs = sorted(set(yte))
        p, r, f, sup = precision_recall_fscore_support(yte, yp, labels=labs, zero_division=0)
        for lab, fi, ri, pi, si in zip(labs, f, r, p, sup):
            perclass.append(dict(seed=seed, cls=lab, f1=fi, recall=ri, precision=pi, support=int(si)))
    df = pd.DataFrame(rows); pc = pd.DataFrame(perclass)
    a = bootstrap_ci(df["accuracy"].values, seed=0); fm = bootstrap_ci(df["macro_f1"].values, seed=1)
    return df, dict(accuracy_mean=a[0], accuracy_lo95=a[1], accuracy_hi95=a[2],
                    macro_f1_mean=fm[0], macro_f1_lo95=fm[1], macro_f1_hi95=fm[2],
                    per_seed_table=df, per_class_table=pc)

def rf_pruned(s): return RandomForestClassifier(n_estimators=15, max_depth=8, max_features=20, random_state=s, n_jobs=-1)
def rf_strong(s): return RandomForestClassifier(n_estimators=200, max_depth=None, max_features="sqrt", random_state=s, n_jobs=-1)
def gbm_builder(s):
    if HAS_XGB:
        return XGBClassifier(n_estimators=400, max_depth=8, learning_rate=0.1,
                             subsample=0.9, colsample_bytree=0.9,
                             tree_method="hist", device="cuda",        # <-- A100: ~20x faster fit than CPU
                             random_state=s, n_jobs=-1, eval_metric="mlogloss")
    return HistGradientBoostingClassifier(max_depth=8, learning_rate=0.1, max_iter=400, random_state=s)


In [ ]:
# ============================================================
# 4) RUN — RF conditions + GBM per variant
#    writes BOTH *_per_seed.csv and *_perclass.csv per cell
# ============================================================
RF_CONDS = {"rf_pruned_fulltrain": (rf_pruned, None),
            "rf_pruned_balanced":  (rf_pruned, BALANCED_PER_CLASS),
            "rf_strong_balanced":  (rf_strong, BALANCED_PER_CLASS)}
ALL_RF, ALL_GBM = {}, {}
for V in VARIANTS:
    print(f"\n=== {V} ===")
    dfv = get_variant(V)
    vout = os.path.join(OUT, f"cic_ids2017_{V}"); os.makedirs(vout, exist_ok=True)
    ALL_RF[V] = {}
    for cname, (b, bd) in RF_CONDS.items():
        _, s = multiseed_eval(dfv, b, bound=bd)
        s["per_seed_table"].to_csv(os.path.join(vout, f"{cname}_per_seed.csv"), index=False)
        s["per_class_table"].to_csv(os.path.join(vout, f"{cname}_perclass.csv"), index=False)
        ALL_RF[V][cname] = s
        print(f"  {cname:<22} acc={s['accuracy_mean']:.4f} [{s['accuracy_lo95']:.4f},{s['accuracy_hi95']:.4f}] macroF1={s['macro_f1_mean']:.4f}")
    gdf, gs = multiseed_eval(dfv, gbm_builder, bound=None, label_encode=True)
    gdf.to_csv(os.path.join(vout, f"{GBM_NAME}_per_seed.csv"), index=False)
    gs["per_class_table"].to_csv(os.path.join(vout, f"{GBM_NAME}_perclass.csv"), index=False)
    ALL_GBM[V] = (GBM_NAME, gdf, gs)
    print(f"  {GBM_NAME:<22} acc={gs['accuracy_mean']:.4f} macroF1={gs['macro_f1_mean']:.4f}  fit={gdf['fit_seconds'].mean():.0f}s")
print("\nDone. Per-seed + per-class CSVs under", os.path.abspath(OUT))

# ---- rare-class F1 readout (the number that drives the paper's banner) ----
print("\n[rare-class F1 - mean over seeds]")
for V in VARIANTS:
    print(f"  {V}:")
    chain = ["rf_pruned_fulltrain", "rf_pruned_balanced", "rf_strong_balanced"]
    for cname in chain:
        pc = ALL_RF[V][cname]["per_class_table"]
        sub = pc[pc["cls"].isin(RARE)].groupby("cls")["f1"].mean()
        if len(sub): print(f"    {cname:<22} " + "  ".join(f"{k}={v:.3f}" for k, v in sub.items()))
    if V in ALL_GBM:
        pc = ALL_GBM[V][2]["per_class_table"]
        sub = pc[pc["cls"].isin(RARE)].groupby("cls")["f1"].mean()
        if len(sub): print(f"    {GBM_NAME:<22} " + "  ".join(f"{k}={v:.3f}" for k, v in sub.items()))



=== V1 ===
Using Colab cache for faster access to the 'cicids2017' dataset.
  raw CIC mirror loaded: (2313810, 78)  cols incl ClassLabel=False
  [V1] (2231806, 70)  classes=15  (pca=False, mapping=False)
  rf_pruned_fulltrain    acc=0.9930 [0.9924,0.9936] macroF1=0.5317
  rf_pruned_balanced     acc=0.9074 [0.9019,0.9141] macroF1=0.4602
  rf_strong_balanced     acc=0.9813 [0.9801,0.9825] macroF1=0.6122


/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [22:55:04] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


  xgboost                acc=0.9987 macroF1=0.8711  fit=38s

=== V2 ===
  [V2] (2231806, 35)  classes=15  (pca=True, mapping=False)
  rf_pruned_fulltrain    acc=0.9881 [0.9873,0.9889] macroF1=0.3875
  rf_pruned_balanced     acc=0.9321 [0.9291,0.9351] macroF1=0.4975
  rf_strong_balanced     acc=0.9652 [0.9634,0.9672] macroF1=0.5644
  xgboost                acc=0.9981 macroF1=0.7922  fit=31s

=== V3 ===
  [V3] (2231806, 70)  classes=9  (pca=False, mapping=True)
  rf_pruned_fulltrain    acc=0.9937 [0.9930,0.9944] macroF1=0.5535
  rf_pruned_balanced     acc=0.9685 [0.9652,0.9713] macroF1=0.5948
  rf_strong_balanced     acc=0.9796 [0.9783,0.9810] macroF1=0.6141
  xgboost                acc=0.9991 macroF1=0.9260  fit=24s

=== V4 ===
  [V4] (2231806, 35)  classes=9  (pca=True, mapping=True)
  rf_pruned_fulltrain    acc=0.9903 [0.9889,0.9913] macroF1=0.4926
  rf_pruned_balanced     acc=0.9520 [0.9454,0.9579] macroF1=0.5420
  rf_strong_balanced     acc=0.9689 [0.9679,0.9700] macroF1=0.5814
  xg

In [ ]:
# ============================================================
# 5) SANITY CHECK — does RF_pruned_fulltrain match the manuscript?
# ============================================================
ref={"V1":0.9947,"V2":0.9889,"V3":0.9952,"V4":0.9901}   # Table-8 single-seed RF
print("Variant  RF_full(mirror)   manuscript   |diff|")
for V in VARIANTS:
    got=ALL_RF[V]["rf_pruned_fulltrain"]["accuracy_mean"]; r=ref[V]
    print(f"  {V}     {got:.4f}          {r:.4f}      {abs(got-r):.4f}"
          + ("   OK" if abs(got-r)<0.01 else "   <-- >1pp, note the mirror difference in the letter"))


Variant  RF_full(mirror)   manuscript   |diff|
  V1     0.9930          0.9947      0.0017   OK
  V2     0.9881          0.9889      0.0008   OK
  V3     0.9937          0.9952      0.0015   OK
  V4     0.9903          0.9901      0.0002   OK


In [ ]:
# ============================================================
# 6) R4.1 DECOMPOSITION (data-condition vs architecture)
# ============================================================
FMS=["tabpfn","tabpfn_2.5","tabicl","tabstar"]
def _fm(V,m):
    p=os.path.join(FM_RUNS,f"cic_ids2017_{V}",f"{m}_summary.json")
    return json.load(open(p)) if os.path.exists(p) else None
rows=[]
for V in VARIANTS:
    full=ALL_RF[V]["rf_pruned_fulltrain"]["accuracy_mean"]
    bal =ALL_RF[V]["rf_pruned_balanced"]["accuracy_mean"]
    strong=ALL_RF[V]["rf_strong_balanced"]["accuracy_mean"]
    rows+=[{"variant":V,"model":"RF_pruned_fulltrain","acc_mean":full,"data_condition_effect":np.nan,"vs_RFbal":np.nan},
           {"variant":V,"model":"RF_pruned_balanced","acc_mean":bal,"data_condition_effect":full-bal,"vs_RFbal":0.0},
           {"variant":V,"model":"RF_strong_balanced","acc_mean":strong,"data_condition_effect":np.nan,"vs_RFbal":strong-bal}]
    for m in FMS:
        s=_fm(V,m)
        if s: rows.append({"variant":V,"model":m,"acc_mean":s["accuracy_mean"],
                           "data_condition_effect":np.nan,"vs_RFbal":s["accuracy_mean"]-bal})
DECOMP=pd.DataFrame(rows); DECOMP.to_csv(os.path.join(OUT,"A1_decomposition.csv"),index=False)
pd.set_option("display.float_format",lambda x:f"{x: .4f}")
print(DECOMP.to_string(index=False))
print("\n'data_condition_effect' = RF_full - RF_balanced (RF's full-distribution advantage).")
print("'vs_RFbal' per FM        = FM - RF_balanced (architecture gap, data condition equalised).")


variant               model  acc_mean  data_condition_effect  vs_RFbal
     V1 RF_pruned_fulltrain    0.9930                    NaN       NaN
     V1  RF_pruned_balanced    0.9074                 0.0856    0.0000
     V1  RF_strong_balanced    0.9813                    NaN    0.0739
     V2 RF_pruned_fulltrain    0.9881                    NaN       NaN
     V2  RF_pruned_balanced    0.9321                 0.0561    0.0000
     V2  RF_strong_balanced    0.9652                    NaN    0.0331
     V3 RF_pruned_fulltrain    0.9937                    NaN       NaN
     V3  RF_pruned_balanced    0.9685                 0.0252    0.0000
     V3  RF_strong_balanced    0.9796                    NaN    0.0111
     V4 RF_pruned_fulltrain    0.9903                    NaN       NaN
     V4  RF_pruned_balanced    0.9520                 0.0383    0.0000
     V4  RF_strong_balanced    0.9689                    NaN    0.0169

'data_condition_effect' = RF_full - RF_balanced (RF's full-distribution adva

In [ ]:
# ============================================================
# 7) R1.3 GBM vs panel + LaTeX snippet
# ============================================================
gbm_rows=[]
for V in VARIANTS:
    name,gdf,gs=ALL_GBM[V]; tp25=_fm(V,"tabpfn_2.5"); ticl=_fm(V,"tabicl")
    gbm_rows.append({"variant":V,"gbm":name,"gbm_acc":gs["accuracy_mean"],"gbm_macro_f1":gs["macro_f1_mean"],
                     "rf_full_acc":ALL_RF[V]["rf_pruned_fulltrain"]["accuracy_mean"],
                     "tabpfn2.5_acc":tp25["accuracy_mean"] if tp25 else np.nan,
                     "tabicl_acc":ticl["accuracy_mean"] if ticl else np.nan,"gbm_fit_s":gdf["fit_seconds"].mean()})
GBM_TBL=pd.DataFrame(gbm_rows); GBM_TBL.to_csv(os.path.join(OUT,"A3_gbm_vs_panel.csv"),index=False)
print(GBM_TBL.to_string(index=False))

def fmt(s): return f"${s['accuracy_mean']:.4f}$ $[{s['accuracy_lo95']:.4f}, {s['accuracy_hi95']:.4f}]$"
lines=[r"% --- new Axis-1 rows (RF control + GBM) ---"]
for V in VARIANTS:
    name,gdf,gs=ALL_GBM[V]
    lines+=[f"% {V}",
            f"  RF (balanced, pruned)  & {fmt(ALL_RF[V]['rf_pruned_balanced'])} \\\\",
            f"  RF (balanced, strong)  & {fmt(ALL_RF[V]['rf_strong_balanced'])} \\\\",
            f"  {name} (full train)    & {fmt(gs)} \\\\"]
open(os.path.join(OUT,"new_rows.tex"),"w").write("\n".join(lines))
print("\n"+"\n".join(lines))


variant     gbm  gbm_acc  gbm_macro_f1  rf_full_acc  tabpfn2.5_acc  tabicl_acc  gbm_fit_s
     V1 xgboost   0.9987        0.8711       0.9930            NaN         NaN    37.8336
     V2 xgboost   0.9981        0.7922       0.9881            NaN         NaN    31.1043
     V3 xgboost   0.9991        0.9260       0.9937            NaN         NaN    23.6762
     V4 xgboost   0.9984        0.8507       0.9903            NaN         NaN    19.6290

% --- new Axis-1 rows (RF control + GBM) ---
% V1
  RF (balanced, pruned)  & $0.9074$ $[0.9019, 0.9141]$ \\
  RF (balanced, strong)  & $0.9813$ $[0.9801, 0.9825]$ \\
  xgboost (full train)    & $0.9987$ $[0.9987, 0.9987]$ \\
% V2
  RF (balanced, pruned)  & $0.9321$ $[0.9291, 0.9351]$ \\
  RF (balanced, strong)  & $0.9652$ $[0.9634, 0.9672]$ \\
  xgboost (full train)    & $0.9981$ $[0.9981, 0.9982]$ \\
% V3
  RF (balanced, pruned)  & $0.9685$ $[0.9652, 0.9713]$ \\
  RF (balanced, strong)  & $0.9796$ $[0.9783, 0.9810]$ \\
  xgboost (full train) 

In [ ]:
# ============================================================
#  CONFIRMATORY CELL — XGBoost on the SAME balanced budget as
#  the foundation models. Run AFTER cell 6 (reuses get_variant,
#  multiseed_eval, gbm_builder, BALANCED_PER_CLASS, VARIANTS, OUT).
#  This nails the data-efficiency claim: XGBoost on ~2k/class
#  should fall toward / below the foundation models on macro-F1.
# ============================================================
import os, pandas as pd

print("XGBoost @ balanced fitting budget (~2k/class) vs full-train:\n")
print(f"{'V':<4} {'XGB_bal_acc':>11} {'XGB_bal_mF1':>12} {'XGB_full_mF1':>13}")
rare_rows = []
for V in VARIANTS:
    dfv = get_variant(V)
    gdf_bal, gs_bal = multiseed_eval(dfv, gbm_builder, bound=BALANCED_PER_CLASS, label_encode=True)
    vout = os.path.join(OUT, f"cic_ids2017_{V}"); os.makedirs(vout, exist_ok=True)
    gdf_bal.to_csv(os.path.join(vout, "xgboost_balanced_per_seed.csv"), index=False)
    gs_bal["per_class_table"].to_csv(os.path.join(vout, "xgboost_balanced_perclass.csv"), index=False)
    # full-train macro-F1 for reference (already on disk)
    full_mf1 = pd.read_csv(os.path.join(vout, "xgboost_per_seed.csv"))["macro_f1"].mean()
    print(f"{V:<4} {gs_bal['accuracy_mean']:>11.4f} {gs_bal['macro_f1_mean']:>12.4f} {full_mf1:>13.4f}")
    # rare-class readout on the balanced budget
    pc = gs_bal["per_class_table"]
    rare = pc[pc["cls"].str.contains("Infiltration|Web Attack|Bot|Heartbleed", regex=True)]
    rr = rare.groupby("cls")["f1"].mean()
    print("      rare-class F1 (bal):  " + "  ".join(f"{k.strip()}={v:.3f}" for k, v in rr.items()))
print("\nCompare XGB_bal_mF1 against the foundation-model macro-F1 in")
print("Table t:multiseed-headline (TabICL/TabPFN2.5 ~0.62-0.72 on the same budget).")

## Notes
- **Faithfulness.** Cell 5 checks `rf_pruned_fulltrain` against the manuscript's RF.
  If within ~1 pp, the Kaggle mirror matches the data the FM runs used and the
  `vs_RFbal` decomposition is sound. If a variant differs by >1 pp, say so in the
  letter (mirror vs academic-server cleaning) — the qualitative R4.1 conclusion
  (RF's lead splits into a data-condition part and a small architecture part) is
  unaffected.
- **R4.1 reading:** `data_condition_effect` (RF_full − RF_balanced) is the slice of
  RF's lead due to its full-distribution access; each FM's `vs_RFbal` is the
  architecture-only gap once the data condition is equalised.
- **GBM:** `xgboost` installs in cell 0; otherwise rows are labelled `hist_gbm`.
- **N-BaIoT / CIC-UNSW:** point `_load_raw` at `mkashifn/nbaiot-dataset` /
  `dhoogla/unswnb15` and adjust the mapping; the RF/GBM machinery is unchanged.


In [ ]:
# ============================================================
#  DOWNLOAD CELL - zips the whole OUT tree (per-seed + per-class),
#  salvages A1_decomposition.csv / new_rows.tex from memory if
#  cells 7-9 were not run, and triggers a Colab download.
# ============================================================
import os, glob, shutil, datetime
import pandas as pd, numpy as np

# OUT defined in cell 2; fall back just in case
try: OUT
except NameError: OUT = "rf_control_and_xgb"

print(f"[scan] {os.path.abspath(OUT)}")
per_seed = sorted(glob.glob(os.path.join(OUT, "**", "*_per_seed.csv"), recursive=True))
by_variant = {}
for p in per_seed:
    V = os.path.basename(os.path.dirname(p)).replace("cic_ids2017_", "")
    by_variant.setdefault(V, []).append(os.path.basename(p).replace("_per_seed.csv", ""))
print(f"\n[variants on disk] {sorted(by_variant) or 'NONE'}")
for V in sorted(by_variant):
    conds = sorted(by_variant[V])
    print(f"   {V}: {len(conds)}/4  {conds}  {'OK' if len(conds) == 4 else '<-- partial'}")

# ---- salvage analysis outputs if dicts still live ----
def _try(fn, label):
    try: fn(); print(f"[salvage] wrote {label}")
    except NameError: print(f"[salvage] {label}: in-memory dicts not found, skipping")
    except Exception as e: print(f"[salvage] {label}: {type(e).__name__}: {e}")

def _decomp():
    rows = []
    for V in sorted(ALL_RF):
        full = ALL_RF[V]["rf_pruned_fulltrain"]["accuracy_mean"]
        bal  = ALL_RF[V]["rf_pruned_balanced"]["accuracy_mean"]
        strg = ALL_RF[V]["rf_strong_balanced"]["accuracy_mean"]
        rows += [
            {"variant": V, "model": "RF_pruned_fulltrain", "acc_mean": full, "data_condition_effect": np.nan, "vs_RFbal": np.nan},
            {"variant": V, "model": "RF_pruned_balanced",  "acc_mean": bal,  "data_condition_effect": full - bal, "vs_RFbal": 0.0},
            {"variant": V, "model": "RF_strong_balanced",  "acc_mean": strg, "data_condition_effect": np.nan, "vs_RFbal": strg - bal},
        ]
        if "ALL_GBM" in globals() and V in ALL_GBM:
            name, _, gs = ALL_GBM[V]
            rows.append({"variant": V, "model": name, "acc_mean": gs["accuracy_mean"], "data_condition_effect": np.nan, "vs_RFbal": gs["accuracy_mean"] - bal})
    pd.DataFrame(rows).to_csv(os.path.join(OUT, "A1_decomposition.csv"), index=False)

def _newrows():
    def fmt(s): return f"${s['accuracy_mean']:.4f}$ $[{s['accuracy_lo95']:.4f}, {s['accuracy_hi95']:.4f}]$"
    L = [r"% --- new Axis-1 rows (RF control + GBM) ---"]
    for V in sorted(ALL_RF):
        L.append(f"% {V}")
        L.append(f"  RF (balanced, pruned)  & {fmt(ALL_RF[V]['rf_pruned_balanced'])} \\\\")
        L.append(f"  RF (balanced, strong)  & {fmt(ALL_RF[V]['rf_strong_balanced'])} \\\\")
        if "ALL_GBM" in globals() and V in ALL_GBM:
            name, _, gs = ALL_GBM[V]
            L.append(f"  {name} (full train)    & {fmt(gs)} \\\\")
    open(os.path.join(OUT, "new_rows.tex"), "w").write("\n".join(L))

_try(_decomp,  "A1_decomposition.csv")
_try(_newrows, "new_rows.tex")

# ---- manifest + zip ----
all_files = [f for f in sorted(glob.glob(os.path.join(OUT, "**", "*"), recursive=True)) if os.path.isfile(f)]
print(f"\n[manifest] {len(all_files)} file(s) under {OUT}/:")
for p in all_files:
    print(f"   {os.path.relpath(p, OUT):42s} {os.path.getsize(p)/1024:7.1f} KB")

ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
zip_path = shutil.make_archive(f"rf_xgb_FULL_{ts}", "zip", root_dir=OUT)
print(f"\n[archive] {zip_path}  ({os.path.getsize(zip_path)/1024:.1f} KB)")

try:
    from google.colab import files
    files.download(zip_path)
    print("[download] triggered")
except Exception as e:
    print(f"[download] not on Colab ({type(e).__name__}); file at {os.path.abspath(zip_path)}")


[scan] /content/rf_control_and_xgb

[variants on disk] ['V1', 'V2', 'V3', 'V4']
   V1: 4/4  ['rf_pruned_balanced', 'rf_pruned_fulltrain', 'rf_strong_balanced', 'xgboost']  OK
   V2: 4/4  ['rf_pruned_balanced', 'rf_pruned_fulltrain', 'rf_strong_balanced', 'xgboost']  OK
   V3: 4/4  ['rf_pruned_balanced', 'rf_pruned_fulltrain', 'rf_strong_balanced', 'xgboost']  OK
   V4: 4/4  ['rf_pruned_balanced', 'rf_pruned_fulltrain', 'rf_strong_balanced', 'xgboost']  OK
[salvage] wrote A1_decomposition.csv
[salvage] wrote new_rows.tex

[manifest] 35 file(s) under rf_control_and_xgb/:
   A1_decomposition.csv                           0.9 KB
   A3_gbm_vs_panel.csv                            0.4 KB
   cic_ids2017_V1/rf_pruned_balanced_per_seed.csv     0.5 KB
   cic_ids2017_V1/rf_pruned_balanced_perclass.csv     5.4 KB
   cic_ids2017_V1/rf_pruned_fulltrain_per_seed.csv     0.5 KB
   cic_ids2017_V1/rf_pruned_fulltrain_perclass.csv     4.2 KB
   cic_ids2017_V1/rf_strong_balanced_per_seed.csv     0.5 KB
   

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

[download] triggered
